In [ ]:
%pip install -q sentence-transformers psycopg2-binary pgvector pandas

In [ ]:
import psycopg2
import pandas as pd
from sentence_transformers import SentenceTransformer
from pgvector.psycopg2 import register_vector

In [ ]:
conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database="Meal_planner",
    user="postgres",
    password="postgres"
)
register_vector(conn)

In [ ]:
with conn.cursor() as cur:
    cur.execute("SELECT version();")
    print(cur.fetchone()[0])

In [ ]:
meals = pd.read_sql_query(
    """
    SELECT
        v.id,
        v.name,
        v.meal_type,
        m.cooking_time,
        m.servings,
        v.calories,
        v.protein,
        v.fat,
        v.carbs,
        v.diet_type,
        v.allergens,
        v.tags,
        m.cost,
        m.goal,
        concat_ws(
            ' ',
            v.rag_text,
            'Цель: ' || m.goal,
            'Тип питания: ' || m.diet_type,
            'Время приготовления: ' || m.cooking_time || ' минут',
            'Цена: ' || m.cost || ' рублей'
        ) AS rag_text
    FROM meal_rag_data AS v
    JOIN meals AS m ON m.id = v.id
    ORDER BY v.meal_type, v.name
    """,
    conn,
)
meals.head()

In [ ]:
print(f"Блюд: {len(meals)}")

In [ ]:
print(meals.loc[0, "rag_text"])

In [ ]:
model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

In [ ]:
embedding = model.encode("блюдо для похудения с высоким содержанием белка")
print(len(embedding))

In [ ]:
embeddings = model.encode(
    meals["rag_text"].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True
)

In [ ]:
print(embeddings.shape)

In [ ]:
embeddings[0][:10]

In [ ]:
with conn.cursor() as cur:
    for meal_id, embedding in zip(meals["id"], embeddings):
        cur.execute(
            """
            UPDATE meals
            SET embedding = %s
            WHERE id = %s
            """,
            (embedding, meal_id)
        )
conn.commit()

In [ ]:
pd.read_sql(
    """
    SELECT
        COUNT(*) AS total,
        COUNT(embedding) AS with_embedding
    FROM meals
    """,
    conn
)

In [ ]:
import re

GOAL_ALIASES = {
    "maintain": "maintain",
    "поддерживать форму": "maintain",
    "поддержать форму": "maintain",
    "сохранить форму": "maintain",
    "lose_weight": "lose_weight",
    "сбросить вес": "lose_weight",
    "похудеть": "lose_weight",
    "gain_weight": "gain_weight",
    "набрать массу": "gain_weight",
    "набрать вес": "gain_weight",
}
GOAL_CALORIES = {"maintain": 1800, "lose_weight": 1400, "gain_weight": 2200}

DIET_ALIASES = {
    "omnivore": "omnivore", "всеядный": "omnivore",
    "vegetarian": "vegetarian", "вегетарианство": "vegetarian", "вегетарианец": "vegetarian",
    "vegan": "vegan", "веганство": "vegan", "веган": "vegan",
    "pescatarian": "pescatarian", "пескетарианство": "pescatarian", "пескетарианец": "pescatarian",
}
DIET_COMPATIBILITY = {
    "omnivore": ["omnivore", "vegetarian", "vegan", "pescatarian"],
    "vegetarian": ["vegetarian", "vegan"], "vegan": ["vegan"],
    "pescatarian": ["pescatarian", "vegetarian", "vegan"],
}
EXCLUSION_ROOTS = {
    "мяс": ["мяс", "курин", "индейк", "говядин", "свин", "баран"],
    "рыб": ["рыб", "лосос", "тунец", "треск", "форел"],
    "молок": ["молок", "молоч", "творог", "йогурт", "сыр", "сливк"],
    "молоч": ["молок", "молоч", "творог", "йогурт", "сыр", "сливк"],
    "орех": ["орех", "миндал", "фундук", "кешью", "грецк"], "арахис": ["арахис"],
    "яйц": ["яйц"], "глютен": ["глютен", "хлеб", "макарон", "пшениц"],
    "пшениц": ["глютен", "хлеб", "макарон", "пшениц"], "соя": ["соя", "соев"],
    "лактоз": ["молок", "молоч", "творог", "йогурт", "сыр", "сливк"],
}

def normalize_goal(value):
    if value is None:
        return None
    value = str(value).strip().casefold()
    return GOAL_ALIASES.get(value, value)

def normalize_diet(value):
    if value is None:
        return "omnivore"
    value = str(value).strip().casefold()
    return DIET_ALIASES.get(value, value)

def allowed_diet_types(value):
    diet = normalize_diet(value)
    return DIET_COMPATIBILITY.get(diet, [diet])

def restriction_patterns(restrictions):
    """Turn free-text restrictions into SQL LIKE patterns for allergens and products."""
    if not restrictions:
        return []
    fragments = re.split(r"[,;\n]+|\s+и\s+", str(restrictions).casefold())
    roots = set()
    for fragment in fragments:
        fragment = re.sub(r"^\s*(без|исключить|исключи|не употреблять|не есть|аллергия на|аллергия)\s+", "", fragment).strip()
        if not fragment:
            continue
        matched = False
        for key, variants in EXCLUSION_ROOTS.items():
            if key in fragment:
                roots.update(variants)
                matched = True
        if not matched:
            roots.add(fragment)
    return [f"%{root}%" for root in sorted(roots)]

def load_profile(email):
    profile = pd.read_sql_query(
        """
        SELECT u.id AS user_id, u.name, u.email, p.goal, p.meals_per_day,
               p.diet_type, p.restrictions, p.budget, p.max_cooking_time, p.preferences
        FROM users AS u
        JOIN profiles AS p ON p.user_id = u.id
        WHERE u.email = %s
        """,
        conn,
        params=(email,),
    )
    if profile.empty:
        raise ValueError(f"Профиль для {email!r} не найден в базе Meal_planner")
    return profile.iloc[0].to_dict()

In [ ]:
profile_email = "ivan@example.com"
profile = load_profile(profile_email)
pd.Series(profile)

In [ ]:
def meal_slots(meals_per_day):
    """Choose meal labels and matching meal_type values available in the database."""
    count = int(meals_per_day or 3)
    if count <= 1:
        return [("Обед", "lunch")]
    if count == 2:
        return [("Завтрак", "breakfast"), ("Ужин", "dinner")]
    slots = [
        ("Завтрак", "breakfast"),
        ("Обед", "lunch"),
        ("Ужин", "dinner"),
    ]
    snack_number = 0
    while len(slots) < count:
        snack_number += 1
        label = "Полдник" if snack_number == 1 else "Перекус"
        if snack_number > 2:
            label = f"Перекус {snack_number - 1}"
        slots.insert(-1, (label, "snack"))
    return slots

slots = meal_slots(profile["meals_per_day"])
slots

In [ ]:
def get_eligible_meals(profile, meal_type):
    """Apply hard profile filters before vector similarity ranking."""
    conditions = [
        "m.status = 'published'",
        "m.embedding IS NOT NULL",
        "m.diet_type = ANY(%s)",
        "m.meal_type = %s",
    ]
    params = [allowed_diet_types(profile.get("diet_type")), meal_type]
    goal = normalize_goal(profile.get("goal"))
    if goal:
        conditions.append("m.goal = %s")
        params.append(goal)
    max_cooking_time = profile.get("max_cooking_time")
    if max_cooking_time is not None and not pd.isna(max_cooking_time):
        conditions.append("m.cooking_time <= %s")
        params.append(int(max_cooking_time))
    excluded = restriction_patterns(profile.get("restrictions"))
    if excluded:
        conditions.append(
            """NOT EXISTS (
                SELECT 1
                FROM unnest(COALESCE(m.allergens, ARRAY[]::text[])) AS a(value)
                WHERE lower(a.value) LIKE ANY(%s::text[])
            )"""
        )
        params.append(excluded)
        conditions.append(
            """NOT EXISTS (
                SELECT 1
                FROM meal_products AS mp
                JOIN products AS p ON p.id = mp.product_id
                WHERE mp.meal_id = m.id
                  AND (
                      lower(p.name) LIKE ANY(%s::text[])
                      OR lower(p.category) LIKE ANY(%s::text[])
                  )
            )"""
        )
        params.extend([excluded, excluded])
    query = f"""
        SELECT
            m.id,
            m.name,
            m.meal_type,
            m.calories,
            m.protein,
            m.fat,
            m.carbs,
            m.cost,
            m.cooking_time,
            m.servings,
            m.goal,
            m.diet_type,
            m.allergens,
            m.tags
        FROM meals AS m
        WHERE {" AND ".join(conditions)}
    """
    return pd.read_sql_query(query, conn, params=params)

def retrieve_candidates(query, profile, meal_type, limit=5):
    """Rank only meals that have already passed goal/diet/restriction filters."""
    eligible = get_eligible_meals(profile, meal_type)
    if eligible.empty:
        return eligible.assign(similarity=pd.Series(dtype=float))
    query_embedding = model.encode(query, normalize_embeddings=True)
    eligible_ids = [str(value) for value in eligible["id"].tolist()]
    ranked = pd.read_sql_query(
        """
        SELECT
            m.id,
            m.name,
            m.meal_type,
            m.calories,
            m.protein,
            m.fat,
            m.carbs,
            m.cost,
            m.cooking_time,
            m.servings,
            m.goal,
            m.diet_type,
            m.allergens,
            m.tags,
            1 - (m.embedding <=> %s::vector) AS similarity
        FROM meals AS m
        WHERE m.id = ANY(%s::uuid[])
          AND m.embedding IS NOT NULL
        ORDER BY m.embedding <=> %s::vector
        LIMIT %s
        """,
        conn,
        params=(query_embedding, eligible_ids, query_embedding, limit),
    )
    return ranked

In [ ]:
def retrieve_candidates_by_slot(profile, extra_request="", candidates_per_type=5):
    groups = []
    uuid_by_temporary_id = {}
    next_id = 1
    goal = normalize_goal(profile.get("goal")) or "без заданной цели"
    preferences = profile.get("preferences") or ""
    for slot_label, meal_type in meal_slots(profile.get("meals_per_day")):
        query_parts = [slot_label, goal, preferences, extra_request]
        query = ". ".join(str(part).strip() for part in query_parts if str(part).strip())
        candidates = retrieve_candidates(
            query,
            profile,
            meal_type=meal_type,
            limit=candidates_per_type,
        )
        slot_candidates = []
        for _, row in candidates.iterrows():
            temporary_id = f"M{next_id:03d}"
            next_id += 1
            uuid_by_temporary_id[temporary_id] = str(row["id"])
            slot_candidates.append({
                "candidate_id": temporary_id,
                "name": row["name"],
                "calories": row["calories"],
                "protein": row["protein"],
                "fat": row["fat"],
                "carbs": row["carbs"],
                "price": row["cost"],
                "cooking_time": row["cooking_time"],
            })
        groups.append({
            "slot": slot_label,
            "meal_type": meal_type,
            "candidates": slot_candidates,
        })
    return groups, uuid_by_temporary_id

candidate_groups, candidate_uuid_map = retrieve_candidates_by_slot(
    profile,
    extra_request="Сбалансированный рацион с подходящим количеством белка",
    candidates_per_type=5,
)
candidate_groups

In [ ]:
def prompt_value(value, suffix=""):
    if value is None or (not isinstance(value, (list, tuple, dict)) and pd.isna(value)):
        return "не указано"
    return f"{value}{suffix}"

def goal_label(value):
    return {
        "maintain": "поддерживать форму",
        "lose_weight": "сбросить вес",
        "gain_weight": "набрать массу",
    }.get(normalize_goal(value), "не указана")

def diet_label(value):
    return {
        "omnivore": "всеядный тип питания",
        "vegetarian": "вегетарианское питание",
        "vegan": "веганское питание",
        "pescatarian": "пескетарианское питание",
    }.get(normalize_diet(value), str(value or "не указан"))

def build_generation_prompt(profile, days, groups, extra_request=""):
    slots = meal_slots(profile.get("meals_per_day"))
    goal = normalize_goal(profile.get("goal"))
    daily_calories = GOAL_CALORIES.get(goal)
    exclusions = profile.get("restrictions") or "не указаны"
    preferences = profile.get("preferences") or "не указаны"
    budget = profile.get("budget")
    budget_text = "без ограничения" if budget is None or pd.isna(budget) else f"{budget} руб./день"
    max_time = profile.get("max_cooking_time")
    time_text = "без ограничения" if max_time is None or pd.isna(max_time) else f"не более {int(max_time)} минут на блюдо"

    profile_lines = [
        f"Цель: {goal_label(goal)}.",
        f"Ориентир по калорийности: около {daily_calories} ккал/день." if daily_calories else "Ориентир по калорийности: не задан.",
        f"Тип питания: {diet_label(profile.get('diet_type'))}.",
        f"Исключить аллергены и продукты: {exclusions}.",
        f"Бюджет на день: {budget_text}.",
        f"Максимальное время готовки: {time_text}.",
        f"Вкусовые предпочтения: {preferences}.",
    ]
    slot_lines = [f"День {day}: " + ", ".join(slot for slot, _ in slots) for day in range(1, int(days) + 1)]
    candidate_sections = []
    for group in groups:
        candidate_sections.append(f"{group['slot']}:")
        if not group["candidates"]:
            candidate_sections.append("  Подходящих блюд не найдено; не придумывай блюда для этого типа.")
            continue
        for item in group["candidates"]:
            candidate_sections.append(
                "  "
                f"[{item['candidate_id']}] {item['name']} — "
                f"КБЖУ: {item['calories']} ккал, Б {item['protein']} г, "
                f"Ж {item['fat']} г, У {item['carbs']} г; "
                f"цена {item['price']} руб.; готовка {item['cooking_time']} мин."
            )

    request_line = f"Дополнительное пожелание: {extra_request}.\n" if extra_request else ""
    return f"""Ты составляешь персональный рацион питания. Используй только блюда-кандидаты ниже.

Параметры плана:
- Дней: {int(days)}.
- Приёмов пищи в день: {len(slots)}.
- Типы и порядок приёмов пищи:
  {', '.join(slot for slot, _ in slots)}.

Профиль пользователя:
{chr(10).join(profile_lines)}
{request_line}
Структура плана по дням:
{chr(10).join(slot_lines)}

Блюда-кандидаты, уже отфильтрованные по цели, типу питания, ограничениям и времени готовки, а затем отсортированные RAG-поиском:
{chr(10).join(candidate_sections)}

Правила:
1. Для каждого дня укажи ровно {len(slots)} приёмов пищи и используй нужный тип приёма.
2. Выбирай только candidate_id из списка для соответствующего типа. Не придумывай блюда, UUID, КБЖУ, цены или время готовки.
3. Используй временные ID вида M001 вместо UUID. Не изменяй и не сокращай эти ID.
4. По возможности соблюдай дневной бюджет и максимальное время готовки; если совместить ограничения не получается, явно сообщи об этом.
5. Учитывай цель, калорийность, вкусовые предпочтения и ограничения профиля. Никогда не включай исключённые аллергены или продукты.
6. Старайся разнообразить блюда между днями.

Верни только JSON такого вида:
{{
  "days": [
    {{
      "day": 1,
      "meals": [
        {{"meal_type": "Завтрак", "candidate_id": "M001"}}
      ]
    }}
  ],
  "notes": []
}}
"""

In [ ]:
planned_days = 7
additional_request = "Сбалансированный рацион, простые блюда из доступных продуктов"
full_prompt = build_generation_prompt(
    profile=profile,
    days=planned_days,
    groups=candidate_groups,
    extra_request=additional_request,
)

In [ ]:
print(full_prompt)